# Qwen3Loop-0.6B: Recursive Intermediate-Calibrated Tool GRPO (RIC-GRPO)
### Treinamento Distribuído em Dual GPU (2x Tesla T4 16GB)

> **Instruções:**
> 1. No menu lateral direito do Kaggle (**Notebook options**):
>    - **Accelerator:** Selecione `GPU T4 x2`.
>    - **Internet:** Ative (`On`).
> 2. Adicione o Dataset privado `qwen3loop-tool-grpo-bundle` em **Add Input**.
> 3. Execute todas as células (`Run All`).

In [ ]:
# 1. Verificar GPUs Tesla T4 ativas
!nvidia-smi

In [ ]:
# 3. Copiar e preparar diretório de trabalho
import os, shutil, zipfile

INPUT_DIR = "/kaggle/input/qwen3loop-tool-grpo-bundle"
if not os.path.exists(INPUT_DIR):
    for d in os.listdir("/kaggle/input"):
        if "qwen3loop" in d.lower():
            INPUT_DIR = os.path.join("/kaggle/input", d)
            break

print("Diretório de entrada detectado:", INPUT_DIR)

# Se o bundle estiver em arquivo zip, extrair
zip_candidates = [f for f in os.listdir(INPUT_DIR) if f.endswith(".zip")]
if zip_candidates:
    zip_path = os.path.join(INPUT_DIR, zip_candidates[0])
    print(f"Extraindo arquivo de pacote {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working/extracted")
    if os.path.exists("/kaggle/working/extracted/model"):
        INPUT_DIR = "/kaggle/working/extracted"
    elif os.path.exists("/kaggle/working/extracted/kaggle_bundle/model"):
        INPUT_DIR = "/kaggle/working/extracted/kaggle_bundle"
    elif os.path.exists("/kaggle/working/extracted/qwen3loop-tool-grpo-bundle/model"):
        INPUT_DIR = "/kaggle/working/extracted/qwen3loop-tool-grpo-bundle"

# Copiar engine para o sys.path do Kaggle
shutil.copytree(os.path.join(INPUT_DIR, "engine", "qwen3loop_python"), "/kaggle/working/engine/qwen3loop_python", dirs_exist_ok=True)
shutil.copy2(os.path.join(INPUT_DIR, "rewards_tool_calling.py"), "/kaggle/working/rewards_tool_calling.py")
shutil.copy2(os.path.join(INPUT_DIR, "train_dual_t4_grpo.py"), "/kaggle/working/train_dual_t4_grpo.py")

os.environ["MODEL_DIR"] = os.path.join(INPUT_DIR, "model")
os.environ["CURRICULUM_DIR"] = os.path.join(INPUT_DIR, "data")
os.environ["OUTPUT_DIR"] = "/kaggle/working/qwen3loop_tool_grpo_final"

print("Ambiente preparado com sucesso! Arquivos em:", INPUT_DIR)

In [ ]:
# 3. Copiar e preparar diretório de trabalho
import os, shutil

INPUT_DIR = '/kaggle/input/qwen3loop-tool-grpo-bundle'
if not os.path.exists(INPUT_DIR):
    # Fallback se o nome da pasta variar no upload
    for d in os.listdir('/kaggle/input'):
        if 'qwen3loop' in d.lower():
            INPUT_DIR = os.path.join('/kaggle/input', d)
            break

print('Diretório de entrada detectado:', INPUT_DIR)

# Copiar engine para o sys.path do Kaggle
shutil.copytree(os.path.join(INPUT_DIR, 'engine', 'qwen3loop_python'), '/kaggle/working/engine/qwen3loop_python', dirs_exist_ok=True)
shutil.copy2(os.path.join(INPUT_DIR, 'rewards_tool_calling.py'), '/kaggle/working/rewards_tool_calling.py')
shutil.copy2(os.path.join(INPUT_DIR, 'train_dual_t4_grpo.py'), '/kaggle/working/train_dual_t4_grpo.py')

os.environ['MODEL_DIR'] = os.path.join(INPUT_DIR, 'model')
os.environ['CURRICULUM_DIR'] = os.path.join(INPUT_DIR, 'data')
os.environ['OUTPUT_DIR'] = '/kaggle/working/qwen3loop_tool_grpo_final'

print('Ambiente preparado com sucesso!')

In [ ]:
# 4. Disparar Treino Paralelo DDP em 2x Tesla T4
# G=6 candidatos por prompt (3 por GPU) com 100 passos densos
!torchrun --nproc_per_node=2 /kaggle/working/train_dual_t4_grpo.py

In [ ]:
# 5. Compactar os adaptadores LoRA treinados e a sonda latente para download rápido (~40 MB)
!cd /kaggle/working && zip -r qwen3loop_tool_grpo_lora_final.zip qwen3loop_tool_grpo_final/
!ls -lh /kaggle/working/qwen3loop_tool_grpo_lora_final.zip
print('\nDownload pronto! Clique com o botão direito no arquivo qwen3loop_tool_grpo_lora_final.zip no menu lateral direito e selecione Download.')